# Stage 4: Independent Evaluation (SFT vs PPO)

import os
import torch
import random
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)
from tqdm import tqdm

# ---------------------------
# Paths
# ---------------------------
SFT_PATH = "./sft_gpt2_dolly/final"
PPO_PATH = "./ppo_gpt2_final/final"
RM_PATH  = "./rm_gpt2_hh/final"

device = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
N_EVAL = 100
MAX_NEW_TOKENS = 80
MAX_PROMPT_LEN = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---------------------------
# Load models
# ---------------------------
print("Loading models...")
tokenizer = AutoTokenizer.from_pretrained(SFT_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

sft_model = AutoModelForCausalLM.from_pretrained(SFT_PATH, local_files_only=True).to(device).eval()
ppo_model = AutoModelForCausalLM.from_pretrained(PPO_PATH, local_files_only=True).to(device).eval()
rm_model  = AutoModelForSequenceClassification.from_pretrained(
    RM_PATH, num_labels=1, problem_type="regression", local_files_only=True
).to(device).eval()

# ---------------------------
# Held-out prompts (fixed seed, never used in training)
# ---------------------------
print("Preparing held-out evaluation set...")
dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
all_prompts = []
for ex in dolly:
    instr = ex["instruction"].strip()
    ctx = ex.get("context", "").strip()
    if ctx:
        text = f"Instruction: {instr}\nContext: {ctx}\nResponse:"
    else:
        text = f"Instruction: {instr}\nResponse:"
    all_prompts.append(text)

# Deterministic held-out set
rng = random.Random(SEED)
rng.shuffle(all_prompts)
eval_prompts = all_prompts[-N_EVAL:]          # last 100 after shuffle
print(f"Using {len(eval_prompts)} held-out prompts")

# ---------------------------
# Generation helper
# ---------------------------
def generate(model, prompt, max_new=MAX_NEW_TOKENS):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LEN).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    # return only the continuation
    return full[len(prompt):].strip()

# ---------------------------
# Reward scoring helper
# ---------------------------
def score_reward(prompt, response):
    text = prompt + " " + response
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=MAX_PROMPT_LEN + MAX_NEW_TOKENS).to(device)
    with torch.no_grad():
        r = rm_model(**inputs).logits.squeeze().item()
    return r

# ---------------------------
# Run evaluation
# ---------------------------
results = []
print("Generating and scoring...")

for i, prompt in enumerate(tqdm(eval_prompts)):
    sft_resp = generate(sft_model, prompt)
    ppo_resp = generate(ppo_model, prompt)

    r_sft = score_reward(prompt, sft_resp)
    r_ppo = score_reward(prompt, ppo_resp)

    results.append({
        "id": i,
        "prompt": prompt[:120] + "..." if len(prompt) > 120 else prompt,
        "sft_response": sft_resp[:200] + ("..." if len(sft_resp) > 200 else ""),
        "ppo_response": ppo_resp[:200] + ("..." if len(ppo_resp) > 200 else ""),
        "reward_sft": r_sft,
        "reward_ppo": r_ppo,
        "reward_diff": r_ppo - r_sft,
        "len_sft": len(sft_resp.split()),
        "len_ppo": len(ppo_resp.split()),
    })

df = pd.DataFrame(results)

# ---------------------------
# Aggregate metrics
# ---------------------------
mean_diff = df["reward_diff"].mean()
win_rate_rm = (df["reward_diff"] > 0).mean()          # PPO higher reward
tie_rate_rm = (df["reward_diff"] == 0).mean()
loss_rate_rm = (df["reward_diff"] < 0).mean()

print("\n" + "="*70)
print("EVALUATION RESULTS (Reward Model scores)")
print("="*70)
print(f"Number of prompts          : {len(df)}")
print(f"Mean reward (SFT)          : {df['reward_sft'].mean():.3f}")
print(f"Mean reward (PPO)          : {df['reward_ppo'].mean():.3f}")
print(f"Mean reward difference     : {mean_diff:+.3f}  (PPO − SFT)")
print(f"RM win rate (PPO > SFT)    : {win_rate_rm*100:.1f}%")
print(f"RM tie rate                : {tie_rate_rm*100:.1f}%")
print(f"RM loss rate (PPO < SFT)   : {loss_rate_rm*100:.1f}%")
print(f"Avg response length (SFT)  : {df['len_sft'].mean():.1f} words")
print(f"Avg response length (PPO)  : {df['len_ppo'].mean():.1f} words")

# Show a few qualitative examples
print("\n" + "-"*70)
print("Sample generations (first 5)")
print("-"*70)
for i in range(min(5, len(df))):
    row = df.iloc[i]
    print(f"\n[{i}] Prompt: {row['prompt']}")
    print(f"SFT ({row['reward_sft']:+.2f}): {row['sft_response']}")
    print(f"PPO ({row['reward_ppo']:+.2f}): {row['ppo_response']}")

# Save full table
csv_path = "/content/drive/MyDrive/Research_Artifacts/rlhf_gpt2/eval_sft_vs_ppo.csv"
df.to_csv(csv_path, index=False)
print(f"\nFull results saved to: {csv_path}")